# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Test Accuracy:", ann_test_acc)

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print("CNN Test Accuracy:", cnn_test_acc)

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(ann_history.history['val_accuracy'], label='ANN Val Acc')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN Validation Accuracy")
plt.legend()
plt.show()

## 📈 Interpretation of the Learning Curves

What to look for in the chart above:

- The CNN validation accuracy line sits clearly above the ANN line. This is the main result of the project.
- The ANN flattens early at a lower accuracy. It treats each image as a flat list of 3072 numbers, so it loses all spatial structure (which pixel is next to which).
- The CNN keeps improving because Conv layers look at small patches of the image and learn edges, shapes, and textures. Pooling then keeps the useful signal and reduces size.
- If training accuracy is much higher than validation accuracy, the model is overfitting. Dropout, BatchNormalization, and data augmentation (used later) help close this gap.

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# Suggested optional run:
# aug_history = aug_cnn_model.fit(x_train_norm, y_train, epochs=10, validation_split=0.1)

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc]
})
comparison

## 📚 Key Concepts Used in This Notebook

- **Normalization (0-255 to 0-1):** Smaller, scaled inputs make gradient updates stable and training faster.
- **Dense layer:** A fully connected layer. Every input connects to every neuron. Good for final classification, weak for raw images.
- **Conv2D:** Slides small filters over the image to detect local patterns like edges and corners. This keeps spatial information.
- **BatchNormalization:** Normalizes the outputs of a layer during training. It speeds up training and makes it more stable.
- **MaxPooling2D:** Keeps the strongest value in each small region. It shrinks the feature maps and keeps the important signal.
- **Dropout:** Randomly turns off some neurons during training so the model does not memorize the training data (less overfitting).
- **Data Augmentation:** Creates flipped, rotated, and zoomed versions of images during training so the model sees more variety and generalizes better.
- **EarlyStopping:** Stops training automatically when validation loss stops improving, and restores the best weights. This saves time and avoids overfitting.
- **Why CNN beats ANN on images:** ANN flattens the image and loses the 2D structure. CNN preserves it and learns features in a hierarchy, so it needs fewer parameters and reaches higher accuracy.

# 🎓 Student Learning Tasks
Try these tasks after understanding the notebook:

### ✅ Beginner Tasks
1. Increase ANN layers and observe performance
2. Change CNN filters from 32→64→128
3. Increase epochs to 20
4. Add **EarlyStopping**
5. Add **data augmentation training**

## ✅ Task 1: Increase ANN Layers and Observe Performance

We make the ANN deeper by adding one more Dense block (512 -> 256 -> 128) with Dropout.
The goal is to check whether a deeper fully-connected network helps when the input is a flat image vector.

In [ ]:
# TASK 1: Deeper ANN
ann_model_v2 = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model_v2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_v2_history = ann_model_v2.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

ann_v2_loss, ann_v2_acc = ann_model_v2.evaluate(x_test_flat, y_test)
print("Deeper ANN Test Accuracy:", ann_v2_acc)

## ✅ Task 2: Scale CNN Filters 32 → 64 → 128

A deeper CNN with a clear filter progression. Early layers use few filters to learn simple edges.
Deeper layers use more filters to learn complex patterns. `padding='same'` keeps the size controlled,
and a BatchNorm block is added after every Conv layer for stable training.

In [ ]:
# TASK 2: Improved CNN with filters 32 -> 64 -> 128
cnn_model_v2 = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model_v2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model_v2.summary()

## ✅ Task 3 & 4: Train for 20 Epochs with EarlyStopping

We train the improved CNN for up to 20 epochs. EarlyStopping watches the validation loss and stops
training when it stops improving for 3 epochs in a row, then restores the best weights.
This covers both the "20 epochs" task and the "EarlyStopping" task.

In [ ]:
# TASK 3 & 4: 20 epochs + EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

cnn_v2_history = cnn_model_v2.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop]
)

cnn_v2_loss, cnn_v2_acc = cnn_model_v2.evaluate(x_test_norm, y_test)
print("Improved CNN (20 epochs + EarlyStopping) Test Accuracy:", cnn_v2_acc)

## ✅ Task 5: Train a CNN with Data Augmentation

We reuse the `data_augmentation` pipeline defined earlier (RandomFlip, RandomRotation, RandomZoom)
and place it at the start of the model. Augmentation is active only during training, so each epoch the
model sees slightly different images. This reduces overfitting and improves generalization.

In [ ]:
# TASK 5: Augmented training run (reuses data_augmentation from the section above)
aug_cnn_train = models.Sequential([
    layers.Input(shape=(32,32,3)),
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_train.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_history = aug_cnn_train.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop]
)

aug_loss, aug_acc = aug_cnn_train.evaluate(x_test_norm, y_test)
print("Augmented CNN Test Accuracy:", aug_acc)

# 📊 Final Comparison: All Model Variants

This table and chart contrast the test accuracy of every model built in this notebook:
the two baselines (ANN, CNN) and the three improved variants from the student tasks.

In [ ]:
# Build the final comparison table
final_comparison = pd.DataFrame({
    "Model": [
        "ANN (baseline)",
        "CNN (baseline)",
        "Deeper ANN (Task 1)",
        "Improved CNN 32-64-128 (Task 2-4)",
        "Augmented CNN (Task 5)"
    ],
    "Test Accuracy": [
        ann_test_acc,
        cnn_test_acc,
        ann_v2_acc,
        cnn_v2_acc,
        aug_acc
    ]
})

final_comparison = final_comparison.sort_values(
    "Test Accuracy", ascending=False
).reset_index(drop=True)

final_comparison

In [ ]:
# Bar chart of the final comparison
plt.figure(figsize=(9,5))
plt.barh(final_comparison["Model"], final_comparison["Test Accuracy"], color='steelblue')
plt.xlabel("Test Accuracy")
plt.title("CIFAR-10: Test Accuracy by Model Variant")
plt.gca().invert_yaxis()
for i, v in enumerate(final_comparison["Test Accuracy"]):
    plt.text(v + 0.005, i, f"{v:.3f}", va='center')
plt.tight_layout()
plt.show()